# Validação da Decodificação — Individuo.decodificar()

Objetivo: validar que a decodificação da matriz de chaves `X` em
permutações por máquina (`ordens_por_maquina`) segue a convenção
definida na Seção 1 da Especificação Técnica: **menor chave = maior
prioridade = executa mais cedo**.

Depende apenas de `src/core/individual.py` (Fase 3).

In [ ]:
import sys
import os

# adiciona a raiz do projeto ao path (notebook está em notebooks/)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))


In [ ]:
import numpy as np

from src.core.individual import Individuo, amostrar_individuo, amostrar_populacao
from src.pbil.probability_matrix import inicializar_matriz_p


## Teste 1 — Decodificação com matriz de chaves conhecida

Monta uma matriz `X` manual de 1 máquina x 4 jobs, onde a ordem
correta já é conhecida de antemão, e confere se `decodificar()`
retorna exatamente essa ordem.

In [ ]:
# 1 máquina, 4 jobs — chaves em ordem "embaralhada" de propósito
# job 0 -> chave 0.8 (prioridade mais baixa, executa por último)
# job 1 -> chave 0.1 (prioridade mais alta, executa primeiro)
# job 2 -> chave 0.5
# job 3 -> chave 0.3
chaves_manual = np.array([[0.8, 0.1, 0.5, 0.3]])

individuo_teste = Individuo(chaves_manual)
ordens = individuo_teste.decodificar()

ordem_esperada = np.array([1, 3, 2, 0])  # job 1 primeiro, job 0 por último

print("Ordem obtida: ", ordens[0])
print("Ordem esperada:", ordem_esperada)

assert np.array_equal(ordens[0], ordem_esperada), "Decodificação não segue a convenção esperada!"
print("\nOK — menor chave decodifica como maior prioridade (executa primeiro).")


## Teste 2 — Decodificação com múltiplas máquinas

Confere que cada linha da matriz `X` (cada máquina) é decodificada
de forma independente das demais.

In [ ]:
# 3 máquinas, 4 jobs — cada máquina com uma ordem diferente de propósito
chaves_multi_maquina = np.array([
    [0.8, 0.1, 0.5, 0.3],   # máquina 0: ordem esperada [1, 3, 2, 0]
    [0.2, 0.9, 0.4, 0.1],   # máquina 1: ordem esperada [3, 0, 2, 1] (empate 0.1 resolvido por posição)
    [0.5, 0.5, 0.1, 0.9],   # máquina 2: ordem esperada [2, 0, 1, 3]
])

individuo_multi = Individuo(chaves_multi_maquina)
ordens_multi = individuo_multi.decodificar()

for indice_maquina, ordem in enumerate(ordens_multi):
    print(f"Máquina {indice_maquina}: {ordem}")

assert len(ordens_multi) == 3, "Número de permutações deve ser igual ao número de máquinas."
for ordem in ordens_multi:
    assert sorted(ordem.tolist()) == [0, 1, 2, 3], "Cada ordem deve ser uma permutação válida dos jobs."

print("\nOK — cada máquina decodificada de forma independente, todas permutações válidas.")


## Teste 3 — Amostragem a partir de P + decodificação

Valida que `amostrar_individuo` / `amostrar_populacao` (a partir da
matriz `P`) sempre geram indivíduos com chaves em `[0, 1]` e que a
decodificação resultante sempre é uma permutação válida por máquina,
independente da aleatoriedade.

In [ ]:
numero_maquinas, numero_jobs = 4, 10
matriz_p_teste = inicializar_matriz_p(numero_maquinas, numero_jobs, valor_inicial=0.5)

gerador_aleatorio = np.random.default_rng(42)
populacao_teste = amostrar_populacao(
    matriz_p_teste, n_pop=20, sigma=0.15,
    distribuicao="normal_truncada", gerador_aleatorio=gerador_aleatorio,
)

for individuo in populacao_teste:
    assert individuo.chaves.min() >= 0.0 and individuo.chaves.max() <= 1.0, \
        "Chaves fora do intervalo [0, 1] após amostragem!"

    ordens = individuo.decodificar()
    assert len(ordens) == numero_maquinas
    for ordem in ordens:
        assert sorted(ordem.tolist()) == list(range(numero_jobs)), \
            "Decodificação de indivíduo amostrado não é uma permutação válida!"

print(f"OK — {len(populacao_teste)} indivíduos amostrados e decodificados com sucesso.")


## Conclusão

A decodificação (`Individuo.decodificar()`) está consistente com a
convenção da Seção 1 da Especificação Técnica em todos os cenários
testados: matriz manual, múltiplas máquinas e amostragem real a
partir de `P`.